In [ ]:
import torch
from PIL import Image
import os
from tqdm import tqdm
import anndata
import scanpy as sc
import pandas as pd

In [ ]:
# ---- Configuration ----
device = "cuda:0"
# UNI2-h weights: see https://github.com/mahmoodlab/UNI
#   pip install "uni @ git+https://github.com/mahmoodlab/UNI.git"
#   download weights from Hugging Face (mahmoodlab/UNI2-h) and set assets_dir below
assets_dir = "/path/to/UNI2-h/ckpts"  # UNI2-h model checkpoint directory

# ---- Paths ----
data_path = "../data/hest1k/her2st/"
tif_path = data_path + 'wsis/'   # H&E whole-slide images
st_path = data_path + "st/"      # Spatial transcriptomics data (.h5ad)

In [ ]:
def get_img_patch_embd(img: Image.Image,
                       adata,
                       samplename,
                       device,
                       save_path=None):
    """Extract UNI2-h patch and tile embeddings for all spots in a slide."""
    from uni import get_encoder
    model_UNI, transform_UNI = get_encoder(enc_name='uni2-h', device=device, assets_dir=assets_dir)

    def get_img_embd(patch: Image.Image, model=model_UNI, transform=transform_UNI):
        base_width = 224
        patch_resized = patch.resize((base_width, base_width), Image.Resampling.LANCZOS)
        img_transformed = transform(patch_resized).unsqueeze(dim=0)
        with torch.inference_mode():
            feature_emb = model(img_transformed.to(device))
        return torch.clone(feature_emb)

    def patch_tiles_embd(patch: Image.Image, tile_size: int = 56):
        embd_dim = 1536  # UNI2-h output dimension
        assert patch.size[0] % tile_size == 0 and patch.size[0] == patch.size[1]
        tile_num_x = patch.size[0] // tile_size
        num_tiles = tile_num_x * tile_num_x
        patch_tile_embd = torch.zeros(num_tiles, embd_dim)
        for i in range(num_tiles):
            x_offset = (i % tile_num_x) * tile_size
            y_offset = (i // tile_num_x) * tile_size
            patch_tile = patch.crop((
                x_offset - tile_size, y_offset - tile_size,
                x_offset + 2 * tile_size, y_offset + 2 * tile_size
            ))
            patch_tile_embd[i, :] = torch.clone(get_img_embd(patch_tile))
        return patch_tile_embd.unsqueeze(0)

    # Process each spot
    radius = 112  # patch size: 224×224
    x = adata.obsm["spatial"][:, 0]
    y = adata.obsm["spatial"][:, 1]
    all_patch_ebd_list = []
    all_patch_tiles_ebd_list = []

    for spot_idx in tqdm(range(len(x))):
        patch = img.crop((
            x[spot_idx] - radius, y[spot_idx] - radius,
            x[spot_idx] + radius, y[spot_idx] + radius
        ))
        all_patch_ebd_list.append(get_img_embd(patch))
        all_patch_tiles_ebd_list.append(patch_tiles_embd(patch))

    all_patch_ebd = torch.cat(all_patch_ebd_list, dim=0)
    all_patch_tiles_ebd = torch.cat(all_patch_tiles_ebd_list, dim=0)
    print(f"Final data size: {all_patch_ebd.shape}, {all_patch_tiles_ebd.shape}")

    if save_path is not None:
        os.makedirs(save_path + "1spot_ebd", exist_ok=True)
        os.makedirs(save_path + "1spot_tiles_ebd_56_3x3", exist_ok=True)
        torch.save(all_patch_ebd.detach().cpu(), save_path + "1spot_ebd/" + samplename + ".pt")
        torch.save(all_patch_tiles_ebd.detach().cpu(), save_path + f"1spot_tiles_ebd_56_3x3/" + samplename + ".pt")

In [ ]:
# ---- Load ST data & intersect common genes across slides ----
adata_list = []
fn_list = ["SPA" + str(i) for i in range(119, 155)]
common_genes = None

for fn in fn_list:
    adata = anndata.read_h5ad(st_path + fn + ".h5ad")
    adata_list.append(adata)
    if common_genes is None:
        common_genes = set(adata.var_names)
    else:
        common_genes &= set(adata.var_names)
    print(fn, adata.shape)

common_genes = sorted(list(common_genes))
print(f"Common genes across all slides: {len(common_genes)}")

for i, fn in enumerate(fn_list):
    adata_list[i] = adata_list[i][:, common_genes].copy()
    print(fn, adata_list[i].shape)

In [ ]:
# ---- Union of highly variable genes across slides ----
union_hvg = set()

for i, fn in enumerate(fn_list):
    adata = adata_list[i].copy()
    sc.pp.filter_cells(adata, min_genes=1)
    sc.pp.filter_genes(adata, min_cells=1)
    sc.pp.normalize_total(adata, inplace=True)
    sc.pp.log1p(adata)
    sc.pp.highly_variable_genes(adata, n_top_genes=2000)
    union_hvg |= set(adata.var_names[adata.var["highly_variable"]])
    print(fn, len(union_hvg))

# Filter out ribosomal and mitochondrial genes
union_hvg = sorted([g for g in union_hvg if not g.startswith(("MT", "mt", "RPS", "RPL"))])
print(f"Union HVG count: {len(union_hvg)}")

In [ ]:
# ---- Build expression matrix for union HVGs across all slides ----
all_count_df = pd.DataFrame(
    adata_list[0][:, union_hvg].X.toarray(),
    columns=union_hvg,
    index=[fn_list[0] + "_" + str(i) for i in range(adata_list[0].shape[0])]
).T

for i in range(1, len(fn_list)):
    adata = adata_list[i]
    df = pd.DataFrame(
        adata[:, union_hvg].X.toarray(),
        columns=union_hvg,
        index=[fn_list[i] + "_" + str(j) for j in range(adata.shape[0])]
    ).T
    all_count_df = pd.concat([all_count_df, df], axis=1)
    print(fn_list[i], adata.shape, all_count_df.shape)

all_count_df.fillna(0, inplace=True)
all_count_df = all_count_df.T

In [ ]:
# ---- Rank genes by mean expression and variance ----
all_gene_order_by_mean = all_count_df.mean(axis=0).sort_values(ascending=False).index
all_gene_order_by_std = all_count_df.std(axis=0).sort_values(ascending=False).index

In [ ]:
# ---- Select top-N genes by mean expression (HEG300) ----
num_genes = 300
selected_genes = sorted(all_gene_order_by_mean[:num_genes])
print(f"Selected genes: {len(selected_genes)}")

In [ ]:
# ---- Save gene list and slide list ----
save_dir = data_path + "processed_data/"
os.makedirs(save_dir, exist_ok=True)

with open(save_dir + "selected_gene_list_300.txt", "w") as f:
    for gene in selected_genes:
        f.write(gene + "\n")

with open(save_dir + "all_slide_list.txt", "w") as f:
    for fn in fn_list:
        f.write(fn + "\n")

In [ ]:
# ---- Extract UNI2-h embeddings for all slides ----
# Note: this step requires GPU and writes ~1.5GB per slide
for i in range(len(fn_list)):
    fn = fn_list[i]
    adata = adata_list[i]
    print(f"Processing {fn}...")
    image = Image.open(tif_path + fn + ".tif")
    get_img_patch_embd(image, adata, fn, device="cuda:0",
                       save_path=data_path + "processed_data/")
    print("#" * 40)